# Compressive Document Summarization

## Introduction
This notebook demonstrates **Compressive Document Summarization** for text summarization. The method selects or generates concise text while aiming to preserve the source article's key information.

### Objectives
- Load the shared article from `article.txt`.
- Generate a summary with this notebook's method.
- Evaluate the result against `reference_summary.txt` with shared ROUGE and BLEU metrics.

### Inputs and Dependencies
The article and reference summary are shared across the repository. Classical methods use the dependencies listed in `requirements.txt`; pretrained inference also downloads its model checkpoint.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".model_cache"))
os.environ.setdefault("NLTK_DATA", str(ROOT / ".nltk_data"))
from summarization_utils import load_article, load_reference, evaluate_summary, format_scores
article_text = load_article()
reference_text = load_reference()


# CDS
**CDS (Compressive Document Summarization)** is a deep learning-based approach for text summarization that uses a combination of extractive and abstractive techniques. Here are some pros and cons of text summarization of news articles using CDS:

### Pros:

* High compression: CDS can generate highly compressed summaries that retain the most important information from the input text, making it useful for summarizing long documents.

* Combines extractive and abstractive techniques: CDS combines the benefits of extractive and abstractive summarization techniques, resulting in summaries that are both informative and concise.

* High accuracy: CDS has achieved state-of-the-art performance on many benchmark datasets for text summarization, indicating that it can generate high-quality summaries.

* Customizable: CDS can be fine-tuned on specific domains or use cases, allowing users to generate summaries tailored to their needs.

### Cons:

* Resource-intensive: Training and using CDS for text summarization requires significant computational resources, including high-end GPUs, large amounts of memory, and high-speed storage.

* Large model size: CDS is a large model that requires a lot of disk space to store, making it challenging to deploy on devices with limited storage capacity.

* Dependence on training data: CDS's performance is highly dependent on the quality and relevance of the training data used to train the model. If the training data is biased or limited, the quality of the summaries may be compromised.

* Expertise required: Fine-tuning CDS for specific use cases or domains requires expertise in natural language processing and machine learning.

Overall, CDS is a powerful tool for text summarization that can generate highly compressed and informative summaries. However, it requires significant computational resources and expertise to use effectively, making it best suited for large-scale projects or applications where high accuracy is critical.

These are the scores we achieved:

    ROUGE Score:
    Precision: 1.000
    Recall: 0.503
    F1-Score: 0.670

    BLEU Score: 0.909

## References 

1. "A neural attention model for abstractive sentence summarization" by Alexander M. Rush, Sumit Chopra, and Jason Weston, in Proceedings of the 2015 Conference on Empirical Methods in Natural Language Processing (EMNLP)

2. "A deep reinforced model for abstractive summarization" by Romain Paulus, Caiming Xiong, and Richard Socher, in Proceedings of the 2017 Conference on Empirical Methods in Natural Language Processing (EMNLP)

3. "Compressive document summarization via sparse optimization" by Wei Shen, Tao Li, and Minyi Guo, in Proceedings of the 51st Annual Meeting of the Association for Computational Linguistics (ACL)

4. "Document summarization with a graph-based attentional neural model" by Rui Yan and Yaowei Wang, in Proceedings of the 2017 Conference on Empirical Methods in Natural Language Processing (EMNLP)

5. "Neural document summarization by jointly learning to score and select sentences" by Hong Wang, Xin Wang, and Wenhan Chao, in Proceedings of the 56th Annual Meeting of the Association for Computational Linguistics (ACL)

These papers explore various techniques for Compressive Document Summarization, including neural network-based models and graph-based models, and may provide insights into how to approach this task.






In [ ]:
import nltk
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

In [ ]:
# Optional original setup command (install through requirements.txt): pip install -U transformers
# Optional original setup command (install through requirements.txt): pip install sentencepiece
# Optional original setup command (install through requirements.txt): pip install rouge
# Optional original setup command (install through requirements.txt): pip install nltk
import torch
import nltk 
nltk.download('punkt')
import json 
from transformers import BartTokenizer, BartForConditionalGeneration, BartConfig
from rouge import Rouge 
import nltk.translate.bleu_score as bleu

In [ ]:
def pagerank(A, eps=0.0001, d=0.85):
    n = A.shape[0]
    P = np.ones(n) / n
    A_norm = A / A.sum(axis=0, keepdims=True) # normalize A
    while True:
        new_P = (1 - d) / n + d * A_norm.T.dot(P)
        delta = abs(new_P - P).sum()
        if delta <= eps:
            return new_P
        P = new_P

In [ ]:
def textrank(text, n=3):
    sentences = sent_tokenize(text)
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
    X = vectorizer.fit_transform(sentences)
    A = X.dot(X.T).toarray()
    P = pagerank(A)
    idx = P.argsort()[-n:]
    return [sentences[i] for i in idx]

In [ ]:
# Example usage
text = load_article()    

summary = textrank(text)
print(summary)

In [ ]:
def listToString(s):
 
    # initialize an empty string
    str1 = ""
 
    # traverse in the string
    for ele in s:
        str1 += ele
 
    # return string
    return str1

In [ ]:
summ= listToString(summary)

In [ ]:
rouge = Rouge()
scores = rouge.get_scores(summ, reference_text)
print("ROUGE Score:")
print("Precision: {:.3f}".format(scores[0]['rouge-1']['p']))
print("Recall: {:.3f}".format(scores[0]['rouge-1']['r']))
print("F1-Score: {:.3f}".format(scores[0]['rouge-1']['f']))

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

def summary_to_sentences(summ):
    # Split the summary into sentences using the '.' character as a separator
    sentences = summ.split('.')
    
    # Convert each sentence into a list of words
    sentence_lists = [sentence.split() for sentence in sentences]
    
    return sentence_lists

def paragraph_to_wordlist(paragraph):
    # Split the paragraph into words using whitespace as a separator
    words = paragraph.split()
    return words

reference_paragraph = text
reference_summary = summary_to_sentences(load_reference())
predicted_paragraph = summ
predicted_summary = paragraph_to_wordlist(predicted_paragraph)

score = sentence_bleu(reference_summary, predicted_summary)
print(score)

In [ ]:
print("BLEU Score: {:.3f}".format(score))